## 1. Check Environment

In [ ]:
import os
import sys
from pathlib import Path

import tensorflow as tf

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Working directory:", Path.cwd())

## 2. Install Dependencies

In [ ]:
%pip install -q -r ../requirements.txt matplotlib boto3 sagemaker

## 3. Configure SageMaker and Paths

Set `S3_DATA_ZIP_URI` if your zipped dataset is stored in S3. If the zip is already in Studio, set `LOCAL_DATA_ZIP_PATH` to that file.

In [ ]:
import boto3
import sagemaker

session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()
bucket = session.default_bucket()

print("Region:", region)
print("Execution role:", role)
print("Default bucket:", bucket)

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_DIR / "src"

S3_DATA_ZIP_URI = None
LOCAL_DATA_ZIP_PATH = PROJECT_DIR / "dataset.zip"
EXTRACT_DIR = PROJECT_DIR / "data"
LOCAL_DATA_DIR = EXTRACT_DIR / "dataset"

OUTPUT_DIR = PROJECT_DIR / "artifacts"
MODEL_OUTPUT_PATH = OUTPUT_DIR / "fresh_vs_rotten_model.keras"
CLASS_NAMES_OUTPUT_PATH = OUTPUT_DIR / "fresh_vs_rotten_class_names.txt"

S3_OUTPUT_PREFIX = "rotten-fruit-detection/artifacts"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Dataset zip:", LOCAL_DATA_ZIP_PATH)
print("Extract dir:", EXTRACT_DIR)
print("Dataset dir:", LOCAL_DATA_DIR)
print("Model output:", MODEL_OUTPUT_PATH)

Expected dataset structure:

```text
dataset/
  train/
    images/
      image1.jpg
      image2.jpg
    _annotations.csv
  valid/
    images/
      image3.jpg
    _annotations.csv
  test/
    images/
      image4.jpg
    _annotations.csv
```

The CSV should have columns like:

```text
filename,width,height,class,xmin,ymin,xmax,ymax
image1.jpg,640,640,good apple,218,183,498,495
image2.jpg,640,640,bad banana,203,183,470,618
```

## 4. Prepare Zipped Dataset

In [ ]:
import zipfile


def split_s3_uri(s3_uri):
    if not s3_uri.startswith("s3://"):
        raise ValueError("S3 URI must start with s3://")
    bucket_name, _, prefix = s3_uri[5:].partition("/")
    return bucket_name, prefix.rstrip("/")


def download_s3_file(s3_uri, local_path):
    s3 = boto3.client("s3")
    bucket_name, key = split_s3_uri(s3_uri)
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(bucket_name, key, str(local_path))


def has_train_test(path):
    path = Path(path)
    child_names = {child.name.lower() for child in path.iterdir() if child.is_dir()}
    return "train" in child_names and "test" in child_names


def find_dataset_root(search_dir):
    search_dir = Path(search_dir)
    if has_train_test(search_dir):
        return search_dir

    for path in search_dir.rglob("*"):
        if path.is_dir() and has_train_test(path):
            return path

    raise ValueError(f"Could not find a folder with train and test inside {search_dir}.")


if S3_DATA_ZIP_URI:
    download_s3_file(S3_DATA_ZIP_URI, LOCAL_DATA_ZIP_PATH)
    print("Downloaded dataset zip from:", S3_DATA_ZIP_URI)

if LOCAL_DATA_ZIP_PATH.exists():
    with zipfile.ZipFile(LOCAL_DATA_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    LOCAL_DATA_DIR = find_dataset_root(EXTRACT_DIR)
    print("Extracted dataset zip to:", EXTRACT_DIR)
else:
    LOCAL_DATA_DIR = find_dataset_root(LOCAL_DATA_DIR)

print("Training dataset root:", LOCAL_DATA_DIR)

## 5. Load Dataset

In [ ]:
possible_src_dirs = [
    SRC_DIR,
    PROJECT_DIR / "src",
    Path.cwd() / "src",
    Path.cwd().parent / "src",
]

SRC_DIR = next((path for path in possible_src_dirs if path.exists()), None)

if SRC_DIR is None:
    raise FileNotFoundError(
        "Could not find the src folder. Make sure PROJECT_DIR points to the "
        "rotten-fruit-detection-ai folder."
    )

sys.path.insert(0, str(SRC_DIR))
print("Using src folder:", SRC_DIR)

from dataset import load_data

train_set, val_set, test_set, class_names = load_data(
    data_dir=LOCAL_DATA_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

print("Classes:", class_names)
print("Number of classes:", len(class_names))

## 6. Build and Train Model

In [ ]:
from model import build_model
from train import train

model = build_model(num_classes=len(class_names))

history = train(
    model=model,
    train_set=train_set,
    val_set=val_set,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

## 7. Plot Training History

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

## 8. Evaluate Test Set

In [ ]:
from train import test_model

test_results = test_model(model, test_set)
test_results

## 9. Save Model and Class Names Locally

In [ ]:
model.save(MODEL_OUTPUT_PATH)
CLASS_NAMES_OUTPUT_PATH.write_text("\n".join(class_names))

print("Saved model to:", MODEL_OUTPUT_PATH)
print("Saved class names to:", CLASS_NAMES_OUTPUT_PATH)

## 10. Upload Artifacts to S3

In [ ]:
model_s3_uri = session.upload_data(
    path=str(MODEL_OUTPUT_PATH),
    bucket=bucket,
    key_prefix=S3_OUTPUT_PREFIX,
)
classes_s3_uri = session.upload_data(
    path=str(CLASS_NAMES_OUTPUT_PATH),
    bucket=bucket,
    key_prefix=S3_OUTPUT_PREFIX,
)

print("Model S3 URI:", model_s3_uri)
print("Class names S3 URI:", classes_s3_uri)

## 11. Test One Image

In [ ]:
import numpy as np
from PIL import Image

TEST_IMAGE_PATH = None

if TEST_IMAGE_PATH is not None:
    image = tf.io.read_file(str(TEST_IMAGE_PATH))
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMAGE_SIZE)
    batch = tf.expand_dims(image, axis=0)

    predictions = model.predict(batch)[0]
    predicted_index = int(np.argmax(predictions))

    display(Image.open(TEST_IMAGE_PATH))
    print("Prediction:", class_names[predicted_index])
    print("Confidence:", float(predictions[predicted_index]))
else:
    print("Set TEST_IMAGE_PATH to test a single image.")